# Árboles y Bosques: ¿Es buen vino o no?

Decision Tree vs Random Forest para clasificar calidad de vino.
Visualización del árbol, overfitting, feature importance y comparación de modelos.

**Dataset:** [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality)  
**Autor:** @aroaxinping  
**Fecha:** Abril 2026

---
## 0. Configuración del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score,
)

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})

ACCENT  = '#e85d04'
ACCENT2 = '#6a9ad4'
ACCENT3 = '#2dc653'
WARN    = '#f4d03f'

print('Entorno listo.')

---
## 1. Datos

| Dataset | Fuente | Registros |
|---|---|---|
| Wine Quality | [UCI ML Repository](https://archive.ics.uci.edu/dataset/186/wine+quality) | ~6500 vinos (red + white) |

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

DATA_PATH = Path('../data/processed/wine_clean.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    es_sintetico = False
    print(f'[OK] Datos cargados: {len(df)} vinos')
else:
    print('[INFO] Datos no encontrados. Generando dataset sintético...')
    print('[TIP]  Ejecuta: python src/fetch_wine.py')
    from fetch_wine import generate_synthetic_wine
    df = generate_synthetic_wine()
    df['is_good'] = (df['quality'] >= 7).astype(int)
    es_sintetico = True

if es_sintetico:
    print('\n⚠️  AVISO: Datos sintéticos.')

FEATURES = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
    'density', 'pH', 'sulphates', 'alcohol',
]
TARGET = 'is_good'

print(f'\nBuenos (quality >= 7): {df[TARGET].sum()} ({df[TARGET].mean()*100:.1f}%)')
print(f'Regulares: {(df[TARGET] == 0).sum()}')
df.head()

---
## 2. Exploración rápida

> **Pregunta:** ¿Qué features químicas distinguen un buen vino de uno regular?

In [ ]:
# 2.1 Distribución de quality
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
df['quality'].value_counts().sort_index().plot(kind='bar', ax=ax, color=ACCENT, edgecolor='#333')
ax.set(xlabel='Quality Score', ylabel='Count', title='Distribución de Quality')

ax = axes[1]
for val, color, label in [(0, ACCENT2, 'Regular (<7)'), (1, ACCENT, 'Bueno (>=7)')]:
    data = df[df[TARGET] == val]['alcohol']
    ax.hist(data, bins=30, alpha=0.6, color=color, label=label, density=True)
ax.set(xlabel='Alcohol (%)', ylabel='Densidad', title='Alcohol por Calidad')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Medias por grupo
comparison = df.groupby(TARGET)[FEATURES].mean().T
comparison.columns = ['Regular', 'Bueno']
comparison['Diferencia %'] = ((comparison['Bueno'] - comparison['Regular']) / comparison['Regular'] * 100).round(1)
print(comparison.sort_values('Diferencia %', ascending=False).to_markdown())

In [ ]:
# 2.3 Train/test split
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

---
## 3. Decision Tree

> **Pregunta:** ¿Un solo árbol de decisión puede clasificar bien la calidad del vino?

In [ ]:
# 3.1 Árbol sin podar (overfitting demo)
dt_deep = DecisionTreeClassifier(random_state=42)
dt_deep.fit(X_train, y_train)

acc_train_deep = accuracy_score(y_train, dt_deep.predict(X_train))
acc_test_deep = accuracy_score(y_test, dt_deep.predict(X_test))

print('--- Árbol sin podar ---')
print(f'  Profundidad: {dt_deep.get_depth()}')
print(f'  Hojas:       {dt_deep.get_n_leaves()}')
print(f'  Acc train:   {acc_train_deep:.3f}')
print(f'  Acc test:    {acc_test_deep:.3f}')
print(f'  Gap:         {acc_train_deep - acc_test_deep:.3f} ← overfitting')

In [ ]:
# 3.2 Árbol podado
dt_pruned = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_pruned.fit(X_train, y_train)

acc_train_pruned = accuracy_score(y_train, dt_pruned.predict(X_train))
acc_test_pruned = accuracy_score(y_test, dt_pruned.predict(X_test))
f1_pruned = f1_score(y_test, dt_pruned.predict(X_test))

print('--- Árbol podado (max_depth=5) ---')
print(f'  Profundidad: {dt_pruned.get_depth()}')
print(f'  Hojas:       {dt_pruned.get_n_leaves()}')
print(f'  Acc train:   {acc_train_pruned:.3f}')
print(f'  Acc test:    {acc_test_pruned:.3f}')
print(f'  F1 test:     {f1_pruned:.3f}')
print(f'  Gap:         {acc_train_pruned - acc_test_pruned:.3f} ← mucho mejor')

In [ ]:
# 3.3 Visualización del árbol podado
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    dt_pruned, ax=ax,
    feature_names=FEATURES,
    class_names=['Regular', 'Bueno'],
    filled=True, rounded=True,
    fontsize=8, proportion=True,
)
ax.set_title('Decision Tree — Wine Quality', fontsize=16, color='#fff')
fig.patch.set_facecolor('#0f0f0f')
plt.tight_layout()
plt.show()

# Reglas en texto
print('\n--- Reglas del árbol (primeras 30 líneas) ---')
rules = export_text(dt_pruned, feature_names=FEATURES)
print('\n'.join(rules.split('\n')[:30]))

In [ ]:
# 3.4 Overfitting por profundidad
depths = range(1, 25)
train_accs, test_accs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test, dt.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs, color=ACCENT, lw=2, marker='o', ms=4, label='Train')
ax.plot(depths, test_accs, color=ACCENT2, lw=2, marker='s', ms=4, label='Test')
ax.fill_between(depths, train_accs, test_accs, alpha=0.1, color=WARN)
best_depth = depths[np.argmax(test_accs)]
ax.axvline(best_depth, ls='--', color=ACCENT3, lw=1.5, label=f'Mejor depth = {best_depth}')
ax.set(xlabel='max_depth', ylabel='Accuracy', title='Overfitting: Train vs Test por Profundidad')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mejor profundidad por test accuracy: {best_depth}')

---
## 4. Random Forest

> **Pregunta:** ¿Un bosque de 100 árboles supera al árbol individual?

In [ ]:
# 4.1 Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print('--- Random Forest (100 árboles) ---')
print(f'  Acc train: {accuracy_score(y_train, rf.predict(X_train)):.3f}')
print(f'  Acc test:  {acc_rf:.3f}')
print(f'  F1 test:   {f1_rf:.3f}')
print(f'  AUC:       {auc_rf:.3f}')
print(f'  OOB score: {rf.oob_score_ if hasattr(rf, "oob_score_") else "N/A"}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Regular', 'Bueno']))

In [ ]:
# 4.2 Feature importance: Decision Tree vs Random Forest
fi_dt = pd.Series(dt_pruned.feature_importances_, index=FEATURES).sort_values()
fi_rf = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.barh(fi_dt.index, fi_dt.values, color=ACCENT, edgecolor='#333')
ax.set(xlabel='Importance', title='Decision Tree')

ax = axes[1]
ax.barh(fi_rf.index, fi_rf.values, color=ACCENT2, edgecolor='#333')
ax.set(xlabel='Importance', title='Random Forest')

plt.suptitle('Feature Importance — Árbol vs Bosque', y=1.01, color='#fff', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 3 features por modelo:')
print(f'  Decision Tree:  {fi_dt.tail(3).index.tolist()[::-1]}')
print(f'  Random Forest:  {fi_rf.tail(3).index.tolist()[::-1]}')

---
## 5. Número de árboles vs rendimiento

> **Pregunta:** ¿A partir de cuántos árboles deja de mejorar el bosque?

In [ ]:
# 5.1 Convergencia del Random Forest
n_trees_range = [1, 5, 10, 25, 50, 75, 100, 150, 200, 300]
rf_accs = []

for n in n_trees_range:
    rf_temp = RandomForestClassifier(n_estimators=n, max_depth=10, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train, y_train)
    rf_accs.append(accuracy_score(y_test, rf_temp.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees_range, rf_accs, color=ACCENT, lw=2, marker='o', ms=6)
ax.axhline(acc_test_pruned, ls='--', color=ACCENT2, lw=1.5, label=f'Decision Tree = {acc_test_pruned:.3f}')
ax.set(xlabel='Número de Árboles', ylabel='Accuracy (Test)', title='Convergencia del Random Forest')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Red vs White

> **Pregunta:** ¿El modelo funciona igual para vino tinto y blanco?

In [ ]:
# 6.1 Métricas por tipo de vino
if 'wine_type' in df.columns:
    test_idx = X_test.index
    df_test = df.loc[test_idx].copy()
    df_test['pred'] = y_pred_rf

    for wtype in ['red', 'white']:
        mask = df_test['wine_type'] == wtype
        if mask.sum() > 0:
            acc = accuracy_score(df_test.loc[mask, TARGET], df_test.loc[mask, 'pred'])
            f1 = f1_score(df_test.loc[mask, TARGET], df_test.loc[mask, 'pred'])
            n = mask.sum()
            print(f'{wtype:6s}: n={n:4d} | Acc={acc:.3f} | F1={f1:.3f}')
else:
    print('Columna wine_type no disponible.')

---
## 7. Síntesis y conclusiones

In [ ]:
# 7.1 Tabla comparativa
resumen = pd.DataFrame([
    {
        'Modelo': 'Decision Tree (sin podar)',
        'Acc Train': f'{acc_train_deep:.3f}',
        'Acc Test': f'{acc_test_deep:.3f}',
        'F1 Test': '—',
        'Overfitting': f'{acc_train_deep - acc_test_deep:.3f}',
    },
    {
        'Modelo': 'Decision Tree (podado, depth=5)',
        'Acc Train': f'{acc_train_pruned:.3f}',
        'Acc Test': f'{acc_test_pruned:.3f}',
        'F1 Test': f'{f1_pruned:.3f}',
        'Overfitting': f'{acc_train_pruned - acc_test_pruned:.3f}',
    },
    {
        'Modelo': 'Random Forest (100 árboles)',
        'Acc Train': f'{accuracy_score(y_train, rf.predict(X_train)):.3f}',
        'Acc Test': f'{acc_rf:.3f}',
        'F1 Test': f'{f1_rf:.3f}',
        'Overfitting': f'{accuracy_score(y_train, rf.predict(X_train)) - acc_rf:.3f}',
    },
])
print(resumen.to_markdown(index=False))

print('\n--- Conclusión ---')
print(f'El Random Forest mejora significativamente al árbol individual.')
print(f'El alcohol es la feature más predictiva de calidad en ambos modelos.')
print(f'El árbol sin podar memoriza el train (acc ≈ 1.0) pero generaliza mal.')
print(f'Podar el árbol reduce overfitting; el bosque lo maneja mejor aún.')
print(f'\nPróximo paso: ¿XGBoost con hyperparameter tuning superará al Random Forest?')